# 🧬 DG Studio — Voz realista con IA (Seed-VC)

Este cuaderno toma la **voz guía** que creó DG Studio (tu letra ya cantada con la melodía) y le pone el **timbre real de tu voz**, usando tu grabación de referencia.

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)** → Guardar.

Luego ejecuta las celdas en orden con ▶ (o *Entorno de ejecución → Ejecutar todas*).

- Modelo: [Seed-VC](https://github.com/Plachtaa/seed-vc) (licencia GPL-3.0). Las canciones que generes son tuyas y puedes monetizarlas.
- ⚠️ Usa **solo tu propia voz** como referencia.


## 1 · Instalar (tarda 2–4 minutos solo la primera vez)

In [ ]:
import subprocess, os
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu or '⚠️ No hay GPU: ve a Entorno de ejecución → Cambiar tipo → GPU (T4)')
if not os.path.exists('/content/seed-vc'):
    !git clone -q https://github.com/Plachtaa/seed-vc /content/seed-vc
%cd /content/seed-vc
# Colab ya trae PyTorch con GPU: instalamos solo lo que falta (sin tocar torch ni numpy)
!grep -viE "^(torch|torchvision|torchaudio|numpy|gradio|FreeSimpleGUI|sounddevice|modelscope|funasr|jiwer)" requirements.txt > req_colab.txt
!pip install -q -r req_colab.txt
print('✅ Listo')

## 2 · Subir tus archivos
Sube los **dos** archivos que descargaste desde DG Studio: `voz_guia.wav` y `mi_voz.wav`.

In [ ]:
from google.colab import files
import shutil
os.makedirs('/content/entrada', exist_ok=True)
subidos = files.upload()
for nombre in subidos:
    destino = 'guia.wav' if 'guia' in nombre.lower() else 'mivoz.wav' if 'voz' in nombre.lower() else nombre
    shutil.move(nombre, f'/content/entrada/{destino}')
print(os.listdir('/content/entrada'))
assert os.path.exists('/content/entrada/guia.wav') and os.path.exists('/content/entrada/mivoz.wav'), 'Faltan voz_guia.wav o mi_voz.wav'

## 3 · Convertir a tu voz
`PASOS`: más pasos = mejor calidad pero más lento (30–50 recomendado). `SEMITONOS`: sube o baja la altura si la voz suena muy aguda o grave.

In [ ]:
PASOS = 40
SEMITONOS = 0
!rm -rf /content/salida
!python inference.py --source /content/entrada/guia.wav --target /content/entrada/mivoz.wav --output /content/salida \
  --diffusion-steps {PASOS} --length-adjust 1.0 --inference-cfg-rate 0.7 \
  --f0-condition True --auto-f0-adjust False --semi-tone-shift {SEMITONOS} --fp16 True
import glob
salida = sorted(glob.glob('/content/salida/*.wav'))
assert salida, 'No se generó el audio: revisa los mensajes de arriba'
shutil.copy(salida[-1], '/content/voz_ia.wav')
from IPython.display import Audio, display
display(Audio('/content/voz_ia.wav'))

## 4 · Descargar
Descarga `voz_ia.wav` y en DG Studio toca **🧬 Voz realista (IA) → 📥 Importar voz IA**. La app la mezcla con tu beat y la puedes bajar en MP3.

In [ ]:
files.download('/content/voz_ia.wav')